<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/inference/predict_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. MOUNT DRIVE

In [1]:
from google.colab import drive
drive.mount('/content/drive')

path_model = '/content/drive/MyDrive/Data/Models/'
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


## 2. IMPORT Library

In [2]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
print(f"TF: {tf.__version__}")

TF: 2.20.0


## 3. DEFINISI CUSTOM LOSS (WAJIB ADA SEBELUM LOAD MODEL)

In [3]:
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name='focal_loss'):
        super().__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true_oh = tf.cast(tf.one_hot(tf.cast(y_true, tf.int32), depth=3), tf.float32)
        y_pred    = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1 - 1e-7)
        ce           = -y_true_oh * tf.math.log(y_pred)
        p_t          = tf.reduce_sum(y_true_oh * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        loss         = self.alpha * focal_weight * tf.reduce_sum(ce, axis=-1)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

print("FocalLoss terdefinisi.")

FocalLoss terdefinisi.


## 4. LOAD MODEL & ARTIFACTS

In [4]:
model = keras.models.load_model(
    path_model + 'worksafe_model_v1.keras',
    custom_objects={'FocalLoss': FocalLoss}
)

with open(path_model + 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(path_model + 'imputer.pkl', 'rb') as f:
    imputer = pickle.load(f)

with open(path_model + 'feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

LABEL_MAP = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
print("Model & artifacts loaded.")
print(f"Jumlah fitur: {len(feature_cols)}")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model & artifacts loaded.
Jumlah fitur: 50


##  5. FUNGSI PREPROCESSING INPUT

In [11]:
def preprocess_input(input_dict):
    df = pd.DataFrame([input_dict])

    # Isi kolom yang tidak ada dengan NaN
    for col in feature_cols:
        if col not in df.columns:
            df[col] = np.nan

    df = df[feature_cols]
    df_imp    = imputer.transform(df)
    df_scaled = scaler.transform(df_imp)
    return df_scaled.astype(np.float32)

## 6. FUNGSI PREDIKSI

In [12]:
def predict_risk(input_dict):
    X     = preprocess_input(input_dict)
    proba = model.predict(X, verbose=0)[0]

    kelas = int(np.argmax(proba))
    label = LABEL_MAP[kelas]

    return {
        'risk_label'    : label,
        'risk_class'    : kelas,
        'confidence'    : round(float(proba[kelas]) * 100, 2),
        'probabilities' : {
            'Low Risk'   : round(float(proba[0]) * 100, 2),
            'Medium Risk': round(float(proba[1]) * 100, 2),
            'High Risk'  : round(float(proba[2]) * 100, 2),
        }
    }

In [9]:
print(feature_cols)

['act_repairing_and_maintaining_mechanical_equipment', 'act_controlling_machines_and_processes', 'skl_operation_and_control', 'skl_equipment_maintenance', 'skl_troubleshooting', 'skl_repairing', 'skl_operations_monitoring', 'act_handling_and_moving_objects', 'act_inspecting_equipment,_structures,_or_materials', 'skl_equipment_selection', 'act_operating_vehicles,_mechanized_devices,_or_equipment', 'act_performing_general_physical_activities', 'act_repairing_and_maintaining_electronic_equipment', 'skl_quality_control_analysis', 'skl_speaking', 'skl_active_listening', 'skl_writing', 'skl_reading_comprehension', 'skl_social_perceptiveness', 'skl_active_learning', 'skl_persuasion', 'act_drafting,_laying_out,_and_specifying_technical_devices,_parts,_and_equipment', 'skl_service_orientation', 'skl_critical_thinking', 'skl_negotiation', 'act_establishing_and_maintaining_interpersonal_relationships', 'act_getting_information', 'skl_learning_strategies', 'skl_judgment_and_decision_making', 'act_

## 7. CONTOH INFERENSI

In [13]:
# Ganti key dan value sesuai nama fitur dari feature_cols
# Bisa cek: print(feature_cols) untuk lihat nama fitur

sample = {fc: np.random.uniform(1, 5) for fc in feature_cols}  # dummy random

result = predict_risk(sample)

print("=" * 40)
print("WorkSafe AI - Hasil Prediksi")
print("=" * 40)
print(f"Label      : {result['risk_label']}")
print(f"Kelas      : {result['risk_class']}")
print(f"Confidence : {result['confidence']}%")
print(f"\nProbabilitas:")
for lbl, prob in result['probabilities'].items():
    bar = '█' * int(prob / 5)
    print(f"  {lbl:<14}: {prob:>5}%  {bar}")

WorkSafe AI - Hasil Prediksi
Label      : High Risk
Kelas      : 2
Confidence : 99.69%

Probabilitas:
  Low Risk      :   0.0%  
  Medium Risk   :  0.31%  
  High Risk     : 99.69%  ███████████████████


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 8. BATCH INFERENCE (OPSIONAL)

In [10]:
# Contoh prediksi dari CSV yang sudah ada

path_processed = '/content/drive/MyDrive/Data/Data/Processed/'
df_infer = pd.read_csv(path_processed + 'test_data.csv').head(10)

results = []
for _, row in df_infer.iterrows():
    inp = {col: row[col] for col in feature_cols if col in df_infer.columns}
    res = predict_risk(inp)
    results.append({
        'risk_label' : res['risk_label'],
        'confidence' : res['confidence'],
        'actual'     : int(row['risk_label']) if 'risk_label' in row else None
    })

df_results = pd.DataFrame(results)
df_results['actual_label'] = df_results['actual'].map(LABEL_MAP)
print("Hasil batch inference (10 sampel pertama):")
display(df_results[['actual_label', 'risk_label', 'confidence']])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Hasil batch inference (10 sampel pertama):


,actual_label,risk_label,confidence
0,High Risk,Medium Risk,100.00
1,Low Risk,Medium Risk,99.76
2,High Risk,Medium Risk,100.00
3,Medium Risk,Medium Risk,100.00
4,Medium Risk,Medium Risk,100.00
5,High Risk,Medium Risk,100.00
6,High Risk,Medium Risk,100.00
7,Low Risk,Medium Risk,94.22
8,Low Risk,Medium Risk,100.00
9,Low Risk,Medium Risk,99.58
